# conti safety — random exploration

just fkingg around with the data before i start the actual runs. not organized at all lol(sry dr yadav😭)

In [ ]:
import json, re
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# had to hardcode this bc jupyter kept complaining abt relative paths
DATA_DIR = Path("../conti/data")
OUT_DIR = Path("../outputs")

In [ ]:
# load advbench subset
rows = []
with open(DATA_DIR / "advbench_subset.jsonl") as f:
    for l in f:
        if l.strip():
            rows.append(json.loads(l))

print(len(rows), "rows")
rows[0]  # just checking format

In [ ]:
# print all prompts quickly
for r in rows:
    print(r['prompt'][:90])

In [ ]:
# testing the keyword regex manually bc i dont trust it
# it was missing 'bomb' earlier which was a disaster

patterns = [
    r"\b(bomb|weapon|explosive|grenade)\b",
    r"\b(how to make|how to build).{0,50}\b(poison|drug|meth|malware)\b",
    r"\b(kill|murder).{0,40}\b(person|someone|president)\b",
]
compiled = [re.compile(p, re.I) for p in patterns]

caught, slipped = [], []
for r in rows:
    p = r.get('prompt', '')
    if any(pat.search(p) for pat in compiled):
        caught.append(p)
    else:
        slipped.append(p)

print(f"caught {len(caught)}, missed {len(slipped)}")

In [ ]:
# what slipped through
for s in slipped[:5]:
    print(">>", s[:100])
    print()

# print(slipped)  # full list if needed

In [ ]:
# check what the actual run produced
m_file = OUT_DIR / "run" / "logs" / "round_metrics.jsonl"

if m_file.exists():
    run_data = [json.loads(l) for l in m_file.read_text().splitlines() if l.strip()]
    for rd in run_data:
        r = rd.get('round')
        acc = rd.get('math_pass1', {}).get('accuracy', '???')
        asr = rd.get('safety_asr', {}).get('aggregate', {}).get('asr_proxy', '???')
        print(f"round {r} | math acc: {acc} | safety asr: {asr}")
else:
    print("no run yet")

## dummy drift plot for the ppt slide
numbers are completely made up, just showing the expected trend. will replace with real data after the cluster run

In [ ]:
# FAKE DATA, for illustration only
rounds_x = [0, 1, 2, 3, 4, 5]
naive     = [0.02, 0.09, 0.21, 0.35, 0.43, 0.57]
v_only    = [0.02, 0.04, 0.06, 0.09, 0.11, 0.13]
v_ewc     = [0.02, 0.02, 0.03, 0.03, 0.04, 0.04]
frozen    = [0.02] * 6

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rounds_x, naive,   'o-', color='tomato',    label='naive continual', lw=2)
ax.plot(rounds_x, v_only,  's-', color='orange',    label='+ verifier only', lw=2)
ax.plot(rounds_x, v_ewc,   '^-', color='steelblue', label='+ verifier + EWC', lw=2)
ax.plot(rounds_x, frozen,  'x--',color='gray',      label='frozen baseline', lw=1)

ax.set_xlabel('self-improvement round')
ax.set_ylabel('ASR (attack success rate)')
ax.set_title('safety drift across arms — PLACEHOLDER')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('dummy_drift.png', dpi=150)
plt.show()

In [ ]:
# token length check bc i was worried short prompts get pad-truncated weirdly
# rough estimate only
lens = [len(r.get('prompt','').split()) * 1.3 for r in rows]
print(f"mean ~{np.mean(lens):.0f} tokens, max ~{np.max(lens):.0f}")
print(f"anything over 256: {sum(1 for l in lens if l > 256)}")
# should be fine for 512 max_length, but double check later

In [ ]:
# thats it for now, will add xstest comparison once i have more rounds logged
# TODO: plot drift per benchmark (advbench vs xstest vs donotanswer) instead of just aggregate